# Python for Data Science

## Lecture 8: Machine learning 1.

### Machine learning methods

Categories:

    I. Supervised learning
        1) Classification
        2) Regression
    II. Unsupervised learning
        1) Clustering
        2) Density estimation        
\+ visualization

Now: I.1) and I.2)

### I.1) Classification

IMDB dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df_review = pd.read_csv('data/IMDB Dataset.csv')
df_review

Taking a sample dataset:

In [ ]:
df_positive = df_review[df_review['sentiment']=='positive'][:1000]
df_negative = df_review[df_review['sentiment']=='negative'][:1000]
df_review_sample = pd.concat([df_positive, df_negative])

Creating train and test datasets

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df_review_sample, test_size=0.33, random_state=42)

Setting independent and dependent variables


In [ ]:
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

Converting text to numerical values

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
train_x_vector = tfidf.fit_transform(train_x)
print(train_x_vector)

Displaying the matrix


In [ ]:
pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out())

Transforming vector $x$

In [ ]:
test_x_vector = tfidf.transform(test_x)
test_x_vector

## I.1. Classification methods
### I.1.a) Support vector machine

In [ ]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

In [ ]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all'])))

### I.1.b) Decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dec_tree = DecisionTreeClassifier()
dec_tree.fit(train_x_vector, train_y)

In [ ]:
print(dec_tree.predict(tfidf.transform(['A good movie'])))
print(dec_tree.predict(tfidf.transform(['An excellent movie'])))
print(dec_tree.predict(tfidf.transform(['I did not like this movie at all'])))

### I.1.c) Naive Bayes


In [ ]:
from sklearn.naive_bayes import MultinomialNB
mnb = MultinomialNB()
mnb.fit(train_x_vector, train_y)

In [ ]:
print(mnb.predict(tfidf.transform(['A good movie'])))
print(mnb.predict(tfidf.transform(['An excellent movie'])))
print(mnb.predict(tfidf.transform(['I did not like this movie at all'])))

### I.1.d) Logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression()
log_reg.fit(train_x_vector, train_y)

In [ ]:
print(log_reg.predict(tfidf.transform(['A good movie'])))
print(log_reg.predict(tfidf.transform(['An excellent movie'])))
print(log_reg.predict(tfidf.transform(['I did not like this movie at all'])))

## Model evaluation
### a) Mean accuracy

In [ ]:
print(svc.score(test_x_vector, test_y))
print(dec_tree.score(test_x_vector, test_y))
print(mnb.score(test_x_vector, test_y))
print(log_reg.score(test_x_vector, test_y))

In [ ]:
# Baseline: ignore the text and always predict the most frequent class.
# Any real model should clearly beat this.
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(train_x_vector, train_y)
print('Baseline accuracy:', dummy.score(test_x_vector, test_y))

### b) F1 score

F1 Score = 2*(Recall * Precision) / (Recall + Precision)

Recall is how many of the actual positive cases were found (recalled), i.e. TP / (TP + FN).

Precision is how many of the returned hits were true positive, i.e. how many of the found were correct hits.

In [ ]:
from sklearn.metrics import f1_score
f1_score(test_y, svc.predict(test_x_vector),
         labels=['positive', 'negative'],
         average=None)


### c) Classification score

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(test_y, 
                            svc.predict(test_x_vector),
                            labels=['positive', 'negative']))

### d) Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
conf_mat = confusion_matrix(test_y, 
                            svc.predict(test_x_vector), 
                            labels=['positive', 'negative'])
conf_mat

### Model tuning

In [ ]:
from sklearn.model_selection import GridSearchCV #set the parameters
parameters = {'C': [1,4,8,16,32] ,'kernel':['linear', 'rbf']}
svc_grid = GridSearchCV(SVC(), parameters, cv=5)

svc_grid.fit(train_x_vector, train_y)

In [ ]:
print(svc_grid.best_params_)
print(svc_grid.best_estimator_)
print('Test accuracy of tuned model:', svc_grid.score(test_x_vector, test_y))

## I.2) Regression

In [ ]:
dataset = pd.read_csv('data/student_scores.csv')
dataset.shape

In [ ]:
dataset.head()


In [ ]:
dataset.describe()

Plotting

In [ ]:
dataset.plot(x='Hours', y='Scores', style='o')
plt.show()

Dependent and independent variables

In [ ]:
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, 1].values



Training and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

### I.2.a) Linear regression (ordinary least squares (OLS))

In [ ]:
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(X_train, y_train)
print(regressor.intercept_)
print(regressor.coef_)

Making predictions

In [ ]:
y_pred = regressor.predict(X_test)
df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
df

Evaluating the model

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
print('Mean Absolute Error:', mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(mean_squared_error(y_test, y_pred)))

### I.2.b) Multiple linear regression

In [ ]:
dataset = pd.read_csv('data/petrol_consumption.csv')
dataset.head()
dataset.describe()

Dependent and independent variables

In [ ]:
X = dataset[['Petrol_tax', 'Average_income', 'Paved_Highways',
       'Population_Driver_licence(%)']]
y = dataset['Petrol_Consumption']

Training and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

Training the model

In [ ]:
regressor = LinearRegression()
regressor.fit(X_train, y_train)

Coefficients

In [ ]:
coeff_df = pd.DataFrame(regressor.coef_, X.columns, columns=['Coefficient'])
coeff_df

Making predictions

In [ ]:
y_pred = regressor.predict(X_test)
df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
df

Evaluating the algorithm

In [ ]:
print('Mean Absolute Error:', mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
# Baseline: predict the training mean for every row. A useful regression model
# must beat this MAE/RMSE, otherwise the features add nothing.
from sklearn.dummy import DummyRegressor
dummy_reg = DummyRegressor(strategy='mean')
dummy_reg.fit(X_train, y_train)
base_pred = dummy_reg.predict(X_test)
print('Baseline MAE:', mean_absolute_error(y_test, base_pred))
print('Baseline RMSE:', np.sqrt(mean_squared_error(y_test, base_pred)))

### I.2.c) LASSO regression

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score #cross-validation
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Lasso penalises coefficients, so features must share a scale. Wrapping the scaler
# and the model in a pipeline refits the scaler inside every CV fold on the training
# rows only, so nothing leaks from the held-out data.
las_mod = make_pipeline(StandardScaler(), Lasso(alpha=0.1))
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)

Model evaluating

In [ ]:
# Cross-validate on the TRAINING set only.
scores = cross_val_score(las_mod, X_train, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)

Forcing scores to be positive

In [ ]:
scores = np.absolute(scores)
print('Mean MAE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))

Predictions

In [ ]:
# Fit on the training set, then evaluate on the untouched test set.
las_mod.fit(X_train, y_train)
y_pred = las_mod.predict(X_test)
df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
df

### I.2.d) Bayesian regression

In [ ]:
from sklearn.linear_model import BayesianRidge
reg = BayesianRidge()
reg.fit(X_train, y_train)

In [ ]:
y_pred=reg.predict(X_test)
df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
df

In [ ]:
reg.coef_

## Source

https://towardsdatascience.com/a-beginners-guide-to-text-classification-with-scikit-learn-632357e16f3a

https://scikit-learn.org/stable/modules/linear_model.html

https://stackabuse.com/linear-regression-in-python-with-scikit-learn

https://machinelearningmastery.com/lasso-regression-with-python/